In [ ]:
# flow

# 1. Read one row from test.xlsx
#         │
#         ▼
# 2. Extract the Regulation Family from the title.
#         │
#         ▼
# 3. Search only the latest 3 Board Meetings (newest → oldest).
#         │
#         ▼
# 4. For each Board Meeting Subject:
#         │
#         ├── Regex can identify the regulation?
#         │       │
#         │       ├── YES → Compare with the query regulation.
#         │       │          ├── Match → Save result.
#         │       │          └── No Match → Ignore.
#         │       │
#         │       └── NO → Ask Mistral:
#         │                "Is this agenda primarily about the queried regulation?"
#         │
#         │                ├── YES → Save result.
#         │                └── NO → Ignore.
#         │
#         ▼
# 5. If a match is found, stop searching.
#    Otherwise, continue to the next recent meeting (up to the latest 3).

In [ ]:
###To scrape board meeting data

In [12]:
import time
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# =====================================================
# INPUT
# =====================================================

YEAR = "2026"

# Meeting to scrape (for now)
MEETING_DAY = "19"
MEETING_MONTH = "June"

URL = "https://www.sebi.gov.in/sebiweb/about/AboutAction.do?doBoardMeeting=yes"

OUTPUT_FILE = "SEBI_Board_Meetings.xlsx"

# =====================================================
# DRIVER
# =====================================================

options = Options()
# options.add_argument("--headless=new")   # Uncomment if needed
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

driver.get(URL)

# =====================================================
# CHANGE YEAR
# =====================================================

while True:

    current_year = wait.until(
        EC.visibility_of_element_located((By.ID, "yearid"))
    ).text.strip()

    print("Current Year:", current_year)

    if current_year == YEAR:
        break

    if int(current_year) > int(YEAR):
        wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".yr-nav-left"))
        ).click()
    else:
        wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".yr-nav-right"))
        ).click()

    time.sleep(1)

# =====================================================
# CLICK REQUIRED MEETING
# =====================================================

meeting = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            f"//li[contains(@class,'dp-item')]//a[contains(.,'{MEETING_DAY}') and contains(.,'{MEETING_MONTH}')]"
        )
    )
)

meeting_date = meeting.text.replace("\n", " ").strip()

print(f"\nScraping Meeting: {meeting_date}")

meeting.click()

time.sleep(2)

# =====================================================
# SCRAPE TABLE
# =====================================================

rows = wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "table tbody tr")
    )
)

records = []

for row in rows:

    cols = row.find_elements(By.TAG_NAME, "td")

    if len(cols) < 4:
        continue

    # -------------------------------------------------
    # Subject
    # -------------------------------------------------

    subject = cols[1].text.strip()

    # -------------------------------------------------
    # Agenda PDF
    # -------------------------------------------------

    agenda_pdf = ""

    try:
        agenda_pdf = cols[2].find_element(By.TAG_NAME, "a").get_attribute("href")
    except:
        pass

    # -------------------------------------------------
    # Decision PDF
    # -------------------------------------------------

    decision_pdf = ""

    try:
        decision_pdf = cols[3].find_element(By.TAG_NAME, "a").get_attribute("href")
    except:
        pass

    records.append(
        {
            "Year": str(YEAR),
            "Meeting Date": meeting_date,
            "Subject": subject,
            "Agenda PDF": agenda_pdf,
            "Decision PDF": decision_pdf,
        }
    )

driver.quit()

# =====================================================
# DATAFRAME
# =====================================================

new_df = pd.DataFrame(records)

# Keep Year as text
new_df["Year"] = new_df["Year"].astype(str)

print(f"\nScraped {len(new_df)} rows.")

# =====================================================
# SAVE / APPEND
# =====================================================

output_path = Path(OUTPUT_FILE)

if output_path.exists():

    existing_df = pd.read_excel(
        output_path,
        dtype={"Year": str}
    )

    # Ensure columns exist
    for col in new_df.columns:
        if col not in existing_df.columns:
            existing_df[col] = ""

    existing_df["Year"] = existing_df["Year"].astype(str)

    combined = pd.concat(
        [existing_df, new_df],
        ignore_index=True
    )

    combined = combined.drop_duplicates(
        subset=[
            "Year",
            "Meeting Date",
            "Subject",
            "Agenda PDF",
        ],
        keep="first"
    )

    added_rows = len(combined) - len(existing_df)

else:

    combined = new_df
    added_rows = len(new_df)

# =====================================================
# WRITE EXCEL
# =====================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    combined.to_excel(writer, index=False)

    worksheet = writer.sheets["Sheet1"]

    # Force Year column to text
    for cell in worksheet["A"][1:]:
        cell.number_format = "@"

print(f"\nAdded {added_rows} new rows.")
print(f"Total rows in Excel: {len(combined)}")
print("\nDone.")

Current Year: 2026

Scraping Meeting: Friday 19 th June

Scraped 10 rows.

Added 0 new rows.
Total rows in Excel: 18

Done.


In [ ]:
###To map amendment regulations from test excel to last board meeting excel

In [1]:
import re
import json
import ollama
import pandas as pd

# =====================================================
# CONFIG
# =====================================================

TEST_EXCEL = "/Users/admin/ai-questionnaire-project/Akshayam_ETL_scripts/Newsletters/Newsletters/Pravartiya/test/test.xlsx"

BOARD_MEETING_EXCEL = "/Users/admin/ai-questionnaire-project/Akshayam_ETL_scripts/Newsletters/Newsletters/Pravartiya/test/SEBI_Board_Meetings.xlsx"

LLM_MODEL = "mistral:latest"

# =====================================================
# PROMPT
# =====================================================


SYSTEM_PROMPT = """
You are an information extraction engine.

Extract the canonical SEBI regulation family name.

Rules:

- Return ONLY valid JSON.
- Do not explain.
- Do not include examples.
- Do not include markdown.
- Do not include any extra text.

Output format:

{
  "regulation_name": "<regulation family name>"
}

Examples

Input:
Securities and Exchange Board of India (Intermediaries) (Amendment) Regulations, 2026

Output:
{
  "regulation_name":"Intermediaries"
}

Input:
Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015

Output:
{
  "regulation_name":"Listing Obligations and Disclosure Requirements"
}
"""

# =====================================================
# LLM - NORMALIZE QUERY
# =====================================================


def normalize_query(title):

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": title,
            },
        ],
        options={
            "temperature": 0,
        },
    )

    text = response["message"]["content"].strip()

    try:
        return json.loads(text)["regulation_name"].strip()

    except Exception:

        # Fallback if the model doesn't return valid JSON
        match = re.search(r"\((.*?)\)", title)
        if match:
            return match.group(1).strip()

        return title


# =====================================================
# SUBJECT PARSER
# =====================================================

def extract_regulation(subject):

    subject = str(subject)

    patterns = [

        r"Securities and Exchange Board of India\s*\((.*?)\)\s*Regulations",

        r"SEBI\s*\((.*?)\)\s*Regulations",

        r"\((.*?)\)\s*Regulations",

    ]

    for pattern in patterns:

        match = re.search(pattern, subject, re.IGNORECASE)

        if match:

            regulation = match.group(1).strip()

            regulation = regulation.replace("(Amendment)", "").strip()

            return regulation

    return None


# =====================================================
# STRICT LITERAL MATCH
# =====================================================

def mentions_regulation_literally(query_regulation, subject):
    """
    Returns True only if the subject text literally contains the
    regulation family name (case-insensitive, whitespace-normalized).

    This is a strict traceability check: an agenda item only counts as
    a match if a human reading the subject line alone would recognize
    it as referring to this regulation family by name. Substantive or
    thematic relatedness (e.g. HVDLE/RPT provisions technically living
    within LODR) is NOT sufficient — the name must actually appear.
    """

    def normalize(text):
        return re.sub(r"\s+", " ", text).strip().casefold()

    return normalize(query_regulation) in normalize(subject)


# =====================================================
# LOAD
# =====================================================

test_df = pd.read_excel(TEST_EXCEL)

board_df = pd.read_excel(
    BOARD_MEETING_EXCEL,
    dtype=str
).fillna("")


# =====================================================
# QUERY
# =====================================================

query_title = test_df.iloc[0]["Title"].strip()

print("\n====================================================")
print("ORIGINAL TITLE")
print("====================================================")
print(query_title)

# query_name = normalize_query(query_title)

match = re.search(r"\((.*?)\)", query_title)
query_name = match.group(1).strip()
print("\n====================================================")
print("NORMALIZED QUERY")
print("====================================================")
print(query_name)

# =====================================================
# LATEST 3 MEETINGS
# =====================================================

meeting_order = (
    board_df[
        [
            "Year",
            "Meeting Date",
        ]
    ]
    .drop_duplicates()
    .iloc[::-1]
    .reset_index(drop=True)
)

recent_meetings = meeting_order.head(3)

print("\n====================================================")
print("RECENT MEETINGS")
print("====================================================")
print(recent_meetings)

# =====================================================
# SEARCH
# =====================================================

found = False

for idx, meeting in recent_meetings.iterrows():

    year = meeting["Year"]
    meeting_date = meeting["Meeting Date"]

    print("\n")
    print("=" * 100)
    print(f"Searching Meeting {idx+1}: {meeting_date} ({year})")
    print("=" * 100)

    meeting_df = board_df[
        (board_df["Year"] == year) &
        (board_df["Meeting Date"] == meeting_date)
    ].copy()

    matches = []

    for _, row in meeting_df.iterrows():

        subject = row["Subject"]

        regulation = extract_regulation(subject)

        print("\nSubject:")
        print(subject)

        print("Extracted Regulation:")
        print(regulation)

        if regulation is not None:

            # Explicit regulation found via regex.
            # Trust regex completely.

            if regulation.casefold() == query_name.casefold():

                matches.append(row)
                print("Match: YES (regex exact match)")

            else:
                print("Match: NO (regex found different regulation)")

            # Whether matched or not, do NOT fall back to literal check
            # or LLM — regex already gave us a definitive regulation name.
            continue

        # Regex could not extract a "(...) Regulations" pattern at all.
        # Only reach here for subjects without that explicit phrasing.

        if mentions_regulation_literally(query_name, subject):

            print("Match: YES (literal regulation name found in subject)")
            matches.append(row)

        else:

            print("Match: NO (regulation name not literally present)")

    if matches:

        print("\n")
        print("=" * 100)
        print("MATCH FOUND")
        print("=" * 100)

        result_df = pd.DataFrame(matches)

        print(
            result_df[
                [
                    "Meeting Date",
                    "Subject",
                    "Agenda PDF",
                    "Decision PDF",
                ]
            ].to_string(index=False)
        )

        found = True
        break

    else:

        print("No match found in this meeting.")

if not found:

    print("\nNo exact match found in the latest 3 meetings.")


ORIGINAL TITLE
Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) (Amendment) Regulations, 2026

NORMALIZED QUERY
Listing Obligations and Disclosure Requirements

RECENT MEETINGS
   Year              Meeting Date
0  2025  Wednesday 17 th December


Searching Meeting 1: Wednesday 17 th December (2025)

Subject:
Amendment to Regulation 39 and 40 of Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015.
Extracted Regulation:
Listing Obligations and Disclosure Requirements
Match: YES (regex exact match)

Subject:
Review of Securities and Exchange Board of India (Stock Brokers) Regulations, 1992 and Addendum.
Extracted Regulation:
Stock Brokers
Match: NO (regex found different regulation)

Subject:
Review of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015 - Clarification regarding the timeline for transfer of unclaimed amount by an entity having listed non-convertible sec

In [ ]:
### use llm to generate "why did sebi amend this regulation?" questions from the regulation summary

In [38]:
import ollama

LLM_MODEL = "mistral:latest"

QUESTION_PROMPT = """
You are an expert in SEBI regulations.

Your task is to generate ONE natural-language question that can be used for RAG retrieval.

Inputs:
1. Regulation Number
2. Amendment Gist

Rules:
- Start with "Why did SEBI..."
- Capture the regulatory intent behind the amendment.
- If it is an Omitted amendment, ask why SEBI omitted that provision.
- If it is an Inserted amendment, ask why SEBI inserted that provision.
- If it is a Substituted amendment, ask why SEBI substituted that provision.
- Mention the Regulation Number.
- Keep the question under 30 words.
- Return ONLY the question.
- Do not explain anything.

Example 1

Regulation Number:
61A

Amendment Gist:
Omitted: The existing proviso requiring quarterly disclosure...

Output:
Why did SEBI omit the quarterly disclosure requirement under Regulation 61A?

Example 2

Regulation Number:
15

Amendment Gist:
Inserted: Inserted a new provision that modifies the applicability of regulations 15 to 27 for high value debt listed entities.

Output:
Why did SEBI insert a new provision modifying the applicability of Regulations 15 to 27 under Regulation 15?
"""


def generate_question(regulation_number, amendment_gist):

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": QUESTION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Regulation Number:
{regulation_number}

Amendment Gist:
{amendment_gist}
""",
            },
        ],
        options={
            "temperature": 0,
        },
    )

    return response["message"]["content"].strip()


# ==========================================================
# EXAMPLES
# ==========================================================

g1 = """
Omitted: The Explanation (2) regarding the determination of 'high value debt listed entities' has been omitted. Prior to its omission, it stated that these entities would be determined on basis of value of principal outstanding of listed debt securities as on March 31, 2021.
"""

# g2 = """
# Inserted: Inserted a new provision that modifies the applicability of regulations 15 to 27 for 'high value debt listed entities'. Once these regulations become applicable, they will continue to apply till the value of outstanding listed debt securities as on March 31 in a year reduces and remains below the specified threshold for a period of three consecutive financial years.
# """

print(generate_question("15", g1))
# print(generate_question("15", g2))

Why did SEBI omit the explanation for determining 'high value debt listed entities' under Regulation 15?


In [ ]:
# Regulation Number: 39
# Footer Number: 407
# Gist of amendment: Substituted: The provision related to the effecting of issuance of letter of confirmation or receipts or advices, as applicable, in cases of loss or old decrepit or worn out certificates or receipts or advices, as applicable, in dematerialised form within a period of thirty days from the date of such lodgement has been substituted. The prior provision required the listed entity to effect issuance of letter of confirmation or receipts or advices, as applicable, within a period of thirty days from the date of lodgement. The new provision requires the listed entity to credit of securities pursuant to investor service requests in relation to subdivision, split, consolidation, renewal, exchanges and issuance of duplicate securities on account of loss or old decrepit or worn out certificates in dematerialised form within a period of thirty days from the date of receipt of such request along with relevant documents.
# Existing provisions of Law prior to amendment: Substituted vide Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026. Prior to its substitution, the sub-regulation (2) read as under: “(2) The listed entity shall effect issuance of letter of confirmation or receipts or advices, as applicable, of subdivision, split, consolidation, renewal, exchanges, endorsements, issuance of duplicates thereof or letter of confirmation or receipts or advices, as applicable, in cases of loss or old decrepit or worn out certificates or receipts or advices, as applicable, in dematerialised form within a period of thirty days from the date of such lodgement.” 80

In [2]:
import ollama

LLM_MODEL = "mistral:latest"

QUESTION_PROMPT = """
You are an expert in SEBI regulations.

Your task is to generate ONE natural-language question that can be used for RAG retrieval.

Inputs:
1. Regulation Number
2. Footer Number
3. Gist of Amendment
4. Existing Provision of Law prior to amendment

Rules:
- Start with "Why did SEBI..."
- Capture the regulatory intent behind the amendment.
- For an Omitted amendment, ask why SEBI omitted that provision.
- For an Inserted amendment, ask why SEBI inserted that provision.
- For a Substituted amendment, ask why SEBI substituted the existing provision with the new requirement.
- Focus on the key regulatory change, not minor wording/details.
- Mention the Regulation Number.
- Do NOT mention the Footer Number.
- Do NOT repeat the full existing provision.
- Keep the question under 30 words.
- Return ONLY the question.
- Do not explain anything.

Example:

Regulation Number:
39

Gist of Amendment:
Substituted: The provision requiring listed entities to issue letters of confirmation
within thirty days has been replaced with a requirement to credit securities in
dematerialised form within thirty days upon receipt of investor service requests.

Existing Provision:
The listed entity shall effect issuance of letter of confirmation or receipts
within thirty days from the date of lodgement.

Output:
Why did SEBI substitute the requirement under Regulation 39 to issue letters of confirmation with a requirement to credit securities in dematerialised form?
"""


def generate_question(
    regulation_number,
    footer_number,
    amendment_gist,
    existing_provision
):
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": QUESTION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Regulation Number:
{regulation_number}

Footer Number:
{footer_number}

Gist of Amendment:
{amendment_gist}

Existing Provision of Law prior to amendment:
{existing_provision}
""",
            },
        ],
        options={
            "temperature": 0,
        },
    )

    return response["message"]["content"].strip()


# ==========================================================
# INPUT
# ==========================================================

amendment_gist = """
Substituted: The provision related to the effecting of issuance of letter of
confirmation or receipts or advices, as applicable, in cases of loss or old
decrepit or worn out certificates or receipts or advices, as applicable, in
dematerialised form within a period of thirty days from the date of such
lodgement has been substituted.

The prior provision required the listed entity to effect issuance of letter
of confirmation or receipts or advices, as applicable, within a period of
thirty days from the date of lodgement.

The new provision requires the listed entity to credit securities pursuant
to investor service requests in relation to subdivision, split, consolidation,
renewal, exchanges and issuance of duplicate securities on account of loss
or old decrepit or worn out certificates in dematerialised form within a
period of thirty days from the date of receipt of such request along with
relevant documents.
"""

existing_provision = """
Substituted vide Securities and Exchange Board of India (Listing Obligations
and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026.

Prior to its substitution, sub-regulation (2) read as under:

"(2) The listed entity shall effect issuance of letter of confirmation or
receipts or advices, as applicable, of subdivision, split, consolidation,
renewal, exchanges, endorsements, issuance of duplicates thereof or letter
of confirmation or receipts or advices, as applicable, in cases of loss or
old decrepit or worn out certificates or receipts or advices, as applicable,
in dematerialised form within a period of thirty days from the date of such
lodgement."
"""


# ==========================================================
# GENERATE
# ==========================================================

question = generate_question(
    regulation_number="39",
    footer_number="407",
    amendment_gist=amendment_gist,
    existing_provision=existing_provision,
)

print(question)

Why did SEBI substitute the requirement under Regulation 39 to effect issuance of letters/receipts/advises for crediting securities in dematerialised form upon investor service requests related to subdivision, split, consolidation, renewal, exchanges, and issuance of duplicates or loss/old decrepit certificates?


In [ ]:
# Regulation Number: 62L
# Footer Number: 607
# Gist of amendment: Inserted: The provision regarding applicability of sub-regulation 607 has been substituted. The original provision stated that the sub-regulation shall be applicable if such sale, disposal or lease of assets is not between two wholly-owned subsidiaries of the HVDLE. The amended provision now states that the sub-regulation shall not be applicable if such sale, disposal or lease of assets is between two wholly-owned subsidiaries of the HVDLE.
# Existing provisions of Law prior to amendment: Inserted vide the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026.

In [9]:
import ollama

LLM_MODEL = "mistral:latest"

QUESTION_PROMPT = """
You are an expert in SEBI regulations.

Your task is to generate ONE natural-language question that can be used for RAG retrieval.

Inputs:
1. Regulation Number
2. Footer Number
3. Gist of Amendment
4. Existing Provision of Law prior to amendment

Rules:
- Start with "Why did SEBI..."
- Capture the regulatory intent behind the amendment.
- For an Omitted amendment, ask why SEBI omitted that provision.
- For an Inserted amendment, ask why SEBI inserted that provision.
- For a Substituted amendment, ask why SEBI substituted the existing provision with the new requirement.
- Focus on the key regulatory change, not minor wording/details.
- Identify the important difference between the previous and amended provision.
- For an amendment that changes applicability, focus on the new applicability or exemption.
- Do not list unnecessary procedural details.
- Mention the Regulation Number.
- Do NOT mention the Footer Number.
- Do NOT repeat the full existing provision.
- Keep the question under 30 words.
- Return ONLY the question.
- Do not explain anything.

Example:

Regulation Number:
62L

Gist of Amendment:
Inserted: The provision regarding applicability of sub-regulation 607 has been
substituted. The original provision stated that the sub-regulation shall be
applicable if such sale, disposal or lease of assets is not between two
wholly-owned subsidiaries of the HVDLE. The amended provision now states
that the sub-regulation shall not be applicable if such sale, disposal or
lease of assets is between two wholly-owned subsidiaries of the HVDLE.

Existing Provision:
The sub-regulation shall be applicable if such sale, disposal or lease of
assets is not between two wholly-owned subsidiaries of the HVDLE.

Output:
Why did SEBI insert an exemption under Regulation 62L for asset transactions between two wholly-owned subsidiaries of an HVDLE?
"""


def generate_question(
    regulation_number,
    footer_number,
    amendment_gist,
    existing_provision
):

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": QUESTION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Regulation Number:
{regulation_number}

Footer Number:
{footer_number}

Gist of Amendment:
{amendment_gist}

Existing Provision of Law prior to amendment:
{existing_provision}
""",
            },
        ],
        options={
            "temperature": 0,
        },
    )

    return response["message"]["content"].strip()


# ==========================================================
# INPUT
# ==========================================================

amendment_gist = """
Inserted: The provision regarding applicability of sub-regulation 607 has
been substituted.

The original provision stated that the sub-regulation shall be applicable
if such sale, disposal or lease of assets is not between two wholly-owned
subsidiaries of the HVDLE.

The amended provision now states that the sub-regulation shall not be
applicable if such sale, disposal or lease of assets is between two
wholly-owned subsidiaries of the HVDLE.
"""

existing_provision = """
Inserted vide the Securities and Exchange Board of India (Listing Obligations
and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026.
"""


# ==========================================================
# GENERATE
# ==========================================================

question = generate_question(
    regulation_number="62L",
    footer_number="607",
    amendment_gist=amendment_gist,
    existing_provision=existing_provision,
)

print(question)

Why did SEBI insert an exemption for asset transactions between two wholly-owned subsidiaries of an HVDLE under the applicability of sub-regulation 607 in Regulation 62L?


In [ ]:
# Regulation Number: 15
# Footer Number: 91
# Gist of amendment: Omitted: The Explanation (2) regarding the determination of 'high value debt listed entities' has been omitted. Prior to its omission, it stated that these entities would be determined on basis of value of principal outstanding of listed debt securities as on March 31, 2021.
# Existing provisions of Law prior to amendment: Omitted vide the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026. Prior to its omission, the Explanation (2) read as under: “Explanation (2) - The ‘high value debt listed entities’ on the date of notification of this amendment would be determined on basis of value of principal outstanding of listed debt securities as on March 31, 2021.”

In [15]:
import ollama
import re

LLM_MODEL = "mistral:latest"

QUESTION_PROMPT = """
You generate short RAG retrieval questions about SEBI amendments.

Your output must contain EXACTLY ONE question.

MANDATORY FORMAT:
Why did SEBI <action> <short description> under Regulation <number>?

RULES:
1. Start exactly with "Why did SEBI..."
2. Mention the Regulation Number.
3. For Omitted:
   "Why did SEBI omit <short description> under Regulation <number>?"
4. For Inserted:
   "Why did SEBI insert <short description> under Regulation <number>?"
5. For Substituted:
   "Why did SEBI substitute <old requirement> with <new requirement> under Regulation <number>?"
6. Keep the description SHORT and at a conceptual level.
7. Do NOT copy detailed wording from the amendment.
8. Do NOT include dates.
9. Do NOT include specific dates such as March 31, 2021.
10. Do NOT include footer numbers.
11. Do NOT mention "Footer".
12. Do NOT mention "vide".
13. Do NOT mention "w.e.f.".
14. Do NOT mention amendment notification names.
15. Do NOT mention the existing provision verbatim.
16. Do NOT add explanations before or after the question.
17. Do NOT use parentheses.
18. Maximum 20 words.
19. Return ONLY the question.
"""


def generate_question(
    regulation_number,
    footer_number,
    amendment_gist,
    existing_provision
):

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": QUESTION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Regulation Number:
{regulation_number}

Footer Number:
{footer_number}

Gist of Amendment:
{amendment_gist}

Existing Provision:
{existing_provision}
""",
            },
        ],
        options={
            "temperature": 0,
        },
    )

    question = response["message"]["content"].strip()

    return clean_question(
        question,
        regulation_number
    )


# ==========================================================
# CLEAN LLM OUTPUT
# ==========================================================

def clean_question(question, regulation_number):

    # Remove markdown/code formatting
    question = question.replace("```", "").strip()

    # Remove quotes
    question = question.strip('"').strip("'").strip()

    # Keep only first question
    if "?" in question:
        question = question.split("?")[0] + "?"

    # Remove parentheses and their contents
    question = re.sub(
        r"\([^)]*\)",
        "",
        question
    )

    # Remove footer references
    question = re.sub(
        r",?\s*Footer\s*(Number)?\s*\d+",
        "",
        question,
        flags=re.IGNORECASE
    )

    # Remove common date patterns
    question = re.sub(
        r"\s+as of\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}",
        "",
        question,
        flags=re.IGNORECASE
    )

    question = re.sub(
        r"\s+on\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}",
        "",
        question,
        flags=re.IGNORECASE
    )

    # Remove "vide..." if model adds it
    question = re.split(
        r"\s+vide\s+",
        question,
        flags=re.IGNORECASE
    )[0]

    # Remove w.e.f.
    question = re.split(
        r"\s+w\.e\.f\.?",
        question,
        flags=re.IGNORECASE
    )[0]

    # Clean duplicate spaces
    question = re.sub(
        r"\s+",
        " ",
        question
    ).strip()

    # Ensure question mark
    if not question.endswith("?"):
        question += "?"

    return question


# ==========================================================
# INPUT
# ==========================================================

amendment_gist = """
Omitted: The Explanation (2) regarding the determination of 'high value debt
listed entities' has been omitted. Prior to its omission, it stated that these
entities would be determined on basis of value of principal outstanding of
listed debt securities as on March 31, 2021.
"""

existing_provision = """
Omitted vide the Securities and Exchange Board of India (Listing Obligations
and Disclosure Requirements) (Amendment) Regulations, 2026 w.e.f. 22.1.2026.

Prior to its omission, the Explanation (2) read as under:

"Explanation (2) - The 'high value debt listed entities' on the date of
notification of this amendment would be determined on basis of value of
principal outstanding of listed debt securities as on March 31, 2021."
"""


# ==========================================================
# GENERATE
# ==========================================================

question = generate_question(
    regulation_number="15",
    footer_number="91",
    amendment_gist=amendment_gist,
    existing_provision=existing_provision,
)

print(question)

Why did SEBI omit the determination of 'high value debt listed entities' based on value of principal outstanding of listed debt securities as under Regulation 15?


In [ ]:
#rag

In [16]:
import fitz
import faiss
import numpy as np
import ollama

from sentence_transformers import SentenceTransformer


# =====================================================
# CONFIG
# =====================================================

PDF_PATH = "/Users/admin/Downloads/1767338412905_1.pdf"

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# Alternatives:
# sentence-transformers/all-MiniLM-L6-v2
# BAAI/bge-small-en-v1.5
# BAAI/bge-base-en-v1.5
# BAAI/bge-large-en-v1.5
# BAAI/bge-m3

LLM_MODEL = "mistral:latest"

TOP_K = 15


# =====================================================
# PDF
# =====================================================

def extract_pdf_pages(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc):

        text = page.get_text("text")

        if text.strip():

            pages.append(
                {
                    "page": page_num + 1,
                    "text": text
                }
            )

    doc.close()

    return pages


# =====================================================
# EMBEDDINGS
# =====================================================

print(f"\nLoading {EMBEDDING_MODEL}")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Loaded.")


def create_embeddings(pages):

    texts = [p["text"] for p in pages]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=16
    )

    return np.array(embeddings).astype("float32")


# =====================================================
# FAISS INDEX
# =====================================================

def build_index(embeddings):

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)

    index.add(embeddings)

    return index


# =====================================================
# RETRIEVE
# =====================================================

def retrieve(question, index, pages):

    # ---------------------------------------------
    # Convert question into embedding
    # ---------------------------------------------

    q_emb = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    q_emb = np.array(q_emb).astype("float32")

    # ---------------------------------------------
    # Search FAISS
    # ---------------------------------------------

    scores, ids = index.search(
        q_emb,
        TOP_K
    )

    results = []

    for score, idx in zip(scores[0], ids[0]):

        results.append(
            {
                "score": float(score),
                "page": pages[idx]["page"],
                "text": pages[idx]["text"]
            }
        )

    return results


# =====================================================
# LLM
# =====================================================

def answer_question(question, retrieved_pages):

    # ---------------------------------------------
    # Build context from retrieved pages
    # ---------------------------------------------

    context_parts = []

    for item in retrieved_pages:

        context_parts.append(
            f"""
==============================
PAGE {item['page']}
Similarity Score: {item['score']:.4f}
==============================

{item['text']}
"""
        )

    context = "\n\n".join(context_parts)

    # ---------------------------------------------
    # Prompt
    # ---------------------------------------------

    prompt = f"""
You are a SEBI regulatory expert.

Answer ONLY from the supplied context.

IMPORTANT RULES:

1. Do not use outside knowledge.
2. If the question refers to Regulation 39, only discuss Regulation 39.
3. If the question refers to Regulation 40, only discuss Regulation 40.
4. Do not mix information from different regulations.
5. If the answer cannot be found in the supplied context, clearly say:
   "The answer is not available in the retrieved context."
6. When answering an amendment question, use the following structure where
   the information is available:

   1. Existing position
   2. Problem identified
   3. Why amendment was proposed
   4. Public consultation feedback
   5. Board decision
   6. Any Board modifications

7. Do not invent or assume information.
8. At the end of the answer, mention the page numbers from the supplied
   context that support the answer.

Context:

{context}

Question:

{question}

Answer:
"""

    # ---------------------------------------------
    # Mistral
    # ---------------------------------------------

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"]


# =====================================================
# PRINT RETRIEVED PAGES
# =====================================================

def print_retrieved_pages(retrieved):

    print("\n")
    print("=" * 100)
    print("RETRIEVED PAGES")
    print("=" * 100)

    for rank, r in enumerate(retrieved, start=1):

        print(
            f"Rank {rank:02d} "
            f"| Page {r['page']} "
            f"| Score = {r['score']:.4f}"
        )


# =====================================================
# PRINT SOURCE PAGES
# =====================================================

def print_source_pages(retrieved):

    source_pages = sorted(
        set(
            r["page"]
            for r in retrieved
        )
    )

    print("\n")
    print("=" * 100)
    print("SOURCE PAGES PROVIDED TO MISTRAL")
    print("=" * 100)

    print(
        "Pages:",
        ", ".join(
            map(str, source_pages)
        )
    )

    print(
        f"\nTotal source pages: {len(source_pages)}"
    )


# =====================================================
# MAIN
# =====================================================

print("\nReading PDF...")

pages = extract_pdf_pages(
    PDF_PATH
)

print(
    f"Pages indexed: {len(pages)}"
)


# =====================================================
# CREATE EMBEDDINGS
# =====================================================

print("\nCreating page embeddings...")

embeddings = create_embeddings(
    pages
)

print(
    f"Embedding shape: {embeddings.shape}"
)


# =====================================================
# BUILD FAISS
# =====================================================

print("\nBuilding FAISS index...")

index = build_index(
    embeddings
)

print(
    f"FAISS index contains {index.ntotal} vectors."
)

print("\nReady.\n")


# =====================================================
# QUESTION LOOP
# =====================================================

while True:

    question = input(
        "\nQuestion: "
    )

    # ---------------------------------------------
    # Exit
    # ---------------------------------------------

    if question.lower().strip() == "exit":
        print("\nExiting...")
        break

    if not question.strip():
        continue

    # ---------------------------------------------
    # RETRIEVE
    # ---------------------------------------------

    retrieved = retrieve(
        question,
        index,
        pages
    )

    # ---------------------------------------------
    # SHOW RETRIEVED PAGES
    # ---------------------------------------------

    print_retrieved_pages(
        retrieved
    )

    # ---------------------------------------------
    # ANSWER
    # ---------------------------------------------

    print("\n")
    print("=" * 100)
    print("ANSWER")
    print("=" * 100)

    answer = answer_question(
        question,
        retrieved
    )

    print(answer)

    # ---------------------------------------------
    # SHOW SOURCE PAGES
    # ---------------------------------------------

    print_source_pages(
        retrieved
    )


Loading BAAI/bge-base-en-v1.5
Loaded.

Reading PDF...
Pages indexed: 9

Creating page embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

Embedding shape: (9, 768)

Building FAISS index...
FAISS index contains 9 vectors.

Ready.





RETRIEVED PAGES
Rank 01 | Page 1 | Score = 0.6908
Rank 02 | Page 4 | Score = 0.6422
Rank 03 | Page 7 | Score = 0.6280
Rank 04 | Page 8 | Score = 0.6140
Rank 05 | Page 3 | Score = 0.6023
Rank 06 | Page 6 | Score = 0.5822
Rank 07 | Page 5 | Score = 0.5790
Rank 08 | Page 2 | Score = 0.5781
Rank 09 | Page 9 | Score = 0.4832
Rank 10 | Page 9 | Score = -340282346638528859811704183484516925440.0000
Rank 11 | Page 9 | Score = -340282346638528859811704183484516925440.0000
Rank 12 | Page 9 | Score = -340282346638528859811704183484516925440.0000
Rank 13 | Page 9 | Score = -340282346638528859811704183484516925440.0000
Rank 14 | Page 9 | Score = -340282346638528859811704183484516925440.0000
Rank 15 | Page 9 | Score = -340282346638528859811704183484516925440.0000


ANSWER
 The provided text does not directly answer your question about why SEBI omitted the determination of 'high value debt listed entities' based on the value of principal outstanding of listed debt securities as under Regulation 15.

KeyboardInterrupt: Interrupted by user

In [ ]:
#  The provided text does not explicitly explain why SEBI omitted the explanation for determining 'high value debt listed entities' under Regulation 15. However, it is possible that this information might be available in other documents or regulations that are not included in the given text. It could also be an oversight in the provided text. For a definitive answer, one would need to refer to the original source of the regulation or consult official SEBI documents or communications.

#  The information provided does not include details about why SEBI omitted an explanation for determining 'high value debt listed entities' under Regulation 15. However, it can be speculated that the reason might be related to the context of the document which appears to focus on amendments to the LODR Regulations regarding handling of unclaimed amounts and bringing it in conformity with the Companies Act and the Rules made thereunder. The document provides details about proposed changes to various regulations, including draft amendment notifications, but it does not include an explanation for determining 'high value debt listed entities' under Regulation 15. This could suggest that this topic was addressed elsewhere or is not relevant to the current discussion. To confirm the reason, more context or additional information would be required.


In [ ]:
### New approach